In [ ]:
# ============================================================
# Imports
# ============================================================
import os
import time
import math
import copy
import random
import pickle
from collections import OrderedDict
from typing import Tuple
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.transforms import v2 as transforms
from torchvision.models import resnet18, ResNet18_Weights


In [ ]:
# ============================================================
# Paths / device
# ============================================================
BASE_PATH = '/kaggle/input/competitions/26-deep-learning-course/'
TRAIN_DIR = os.path.join(BASE_PATH, 'train')
TEST_DIR  = os.path.join(BASE_PATH, 'test')
train_df  = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device}, gpus={torch.cuda.device_count()}")


In [ ]:
# ============================================================
# Config — single-session pipeline
#   Phase A: ResNet18 (ImageNet pretrained) → CIFAR-100 fine-tune
#   Phase B: A1W1ResNet18v2 student + KD from Phase-A teacher
# ============================================================
seed = 42

# ----- Phase A: teacher ---------------------------------------------
teacher_epochs          = 80
teacher_lr              = 1e-3
teacher_weight_decay    = 1e-4
teacher_label_smoothing = 0.05
teacher_mixup_alpha     = 1.0      # decays via "step"
teacher_min_alpha       = 0.0625
teacher_warmup_epochs   = max(int(teacher_epochs * 0.1), 1)

# ----- Phase B: A1W1 student ----------------------------------------
student_epochs          = 220
student_lr              = 5e-4
student_weight_decay    = 0.0
student_label_smoothing = 0.0
student_grad_clip       = 5.0
student_kd_T            = 4.0
student_kd_weight       = 0.7
student_feature_kd_w    = 0.0      # Feature KD disabled (A2 reverted — BN running stats unstable post-KD switch)
student_feature_kd_ramp = 10       # unused when weight=0
student_mixup_alpha     = 0.4
student_min_alpha       = 0.0
student_warmup_epochs   = 15
student_use_rprelu      = True
student_use_double_skip = True

# ----- Shared knobs --------------------------------------------------
batch_size       = 1024
img_resize       = 32
train_val_split  = 0.90
train_num_workers = 2
min_lr           = 1e-6
mixup_decay      = "step"        # cosine | linear | step | random | none

# Augmentation
aug_num_ops      = 2
aug_magnitude    = 9
re_prob          = 0.2
re_value         = 'random'

ema_decay        = 0.999

# Skip Phase A if a teacher checkpoint already exists (saves ~1.5h on reruns).
# Set False to force re-training the teacher from scratch.
skip_phase_a_if_ckpt = True
teacher_ckpt_path    = "best_teacher.pth"

# Submission
submission_ckpt_path = "best_student.pth"

# Normalize stats (ImageNet, since teacher backbone is ImageNet-pretrained)
CIFAR100_MEAN = [0.4850, 0.4560, 0.4060]
CIFAR100_STD  = [0.2290, 0.2240, 0.2250]

# ----- Experiment id (history 파일 분리 저장) -----
# 변경 예시: "e1_kd0", "e2_singleskip", "e3_no_rprelu"
exp_id = "baseline"


In [ ]:
# ============================================================
# Reproducibility
# ============================================================
def set_seed(s):
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    np.random.seed(s)
    random.seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


In [ ]:
# ============================================================
# Dataset — RAM-resident cache for fast Kaggle training
#   • One-time preload via ThreadPool: PIL.Image objects kept in RAM
#   • Cache keyed by (img_dir, img_resize, is_test) → re-instantiating the
#     loader is free (cache hit) so Phase A and Phase B share images
#   • Heavy augmentation runs on GPU (Cell 5), CPU side does only ToImage()
# ============================================================
if '_IMAGE_CACHE' not in globals():
    _IMAGE_CACHE = {}

class CIFAR100Dataset(Dataset):
    def __init__(self, img_dir, df=None, transform=None, img_resize=32, is_test=False):
        self.img_dir   = img_dir
        self.transform = transform
        self.is_test   = is_test
        self.img_resize = img_resize

        if is_test:
            # Test images MUST be sorted numerically for submission alignment.
            self.img_names = sorted(os.listdir(img_dir), key=lambda x: int(x.split('.')[0]))
            self.labels    = None
        else:
            self.img_names = df['id'].astype(str).tolist()
            self.labels    = df['label'].values

        cache_key = (img_dir, img_resize, is_test)
        if cache_key in _IMAGE_CACHE:
            self.images = _IMAGE_CACHE[cache_key]
            print(f">>> [Cache hit] {len(self.images)} images in RAM ({img_dir})")
        else:
            t0 = time.time()
            print(f">>> RAM preload: {len(self.img_names)} images "
                  f"(resize={img_resize}, dir={img_dir})")

            def _load(name):
                if not name.endswith('.png'):
                    name = name + '.png'
                img = Image.open(os.path.join(img_dir, name)).convert('RGB')
                if img_resize != 32:
                    img = img.resize((img_resize, img_resize), Image.BICUBIC)
                return img

            with ThreadPoolExecutor() as ex:
                self.images = list(ex.map(_load, self.img_names))
            _IMAGE_CACHE[cache_key] = self.images
            print(f">>> Preload done in {time.time() - t0:.1f}s")

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img
        return img, int(self.labels[idx])


In [ ]:
# ============================================================
# Transforms + DataLoader factories
# ============================================================
def worker_init_fn(worker_id):
    s = torch.initial_seed() % (2**32)
    np.random.seed(s)
    random.seed(s)

def get_gpu_transforms():
    train_tf = nn.Sequential(
        transforms.Pad(4, padding_mode='reflect'),
        transforms.RandomCrop(img_resize),
        transforms.RandomHorizontalFlip(),
        transforms.RandAugment(num_ops=aug_num_ops, magnitude=aug_magnitude),
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Normalize(mean=CIFAR100_MEAN, std=CIFAR100_STD),
        transforms.RandomErasing(p=re_prob, value=re_value),
    )
    val_tf = nn.Sequential(
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Normalize(mean=CIFAR100_MEAN, std=CIFAR100_STD),
    )
    return train_tf, val_tf

class _TransformedSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset, self.transform = subset, transform
    def __getitem__(self, i):
        x, y = self.subset[i]
        if self.transform:
            x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.subset)

def train_val_loader():
    cpu_tf = transforms.Compose([transforms.ToImage()])
    base = CIFAR100Dataset(TRAIN_DIR, df=train_df, transform=None, img_resize=img_resize)
    g = torch.Generator().manual_seed(42)
    n_train = int(train_val_split * len(base))
    n_val   = len(base) - n_train
    ids = list(range(len(base)))
    tr_idx, va_idx = torch.utils.data.random_split(ids, [n_train, n_val], generator=g)
    tr_sub = torch.utils.data.Subset(base, tr_idx)
    va_sub = torch.utils.data.Subset(base, va_idx)
    trainset = _TransformedSubset(tr_sub, cpu_tf)
    valset   = _TransformedSubset(va_sub, cpu_tf)
    common = dict(batch_size=batch_size, num_workers=train_num_workers,
                  pin_memory=torch.cuda.is_available(),
                  worker_init_fn=worker_init_fn, persistent_workers=True)
    return (DataLoader(trainset, shuffle=True,  **common),
            DataLoader(valset,   shuffle=False, **common))

def test_loader():
    cpu_tf = transforms.Compose([transforms.ToImage()])
    testset = CIFAR100Dataset(TEST_DIR, transform=cpu_tf, img_resize=img_resize, is_test=True)
    return DataLoader(testset, batch_size=batch_size, shuffle=False,
                      num_workers=train_num_workers,
                      pin_memory=torch.cuda.is_available())


In [ ]:
# ============================================================
# Mixup / CutMix utilities + training & eval loops
# ============================================================
def mixup_data(x, y, lam):
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def cutmix_data(x, y, lam):
    B, _, H, W = x.size()
    idx = torch.randperm(B, device=x.device)
    cut_rat = torch.sqrt(1.0 - lam)
    cw = (W * cut_rat).to(torch.int32); ch = (H * cut_rat).to(torch.int32)
    cx = torch.randint(0, W, (1,), device=x.device)
    cy = torch.randint(0, H, (1,), device=x.device)
    x1 = torch.clamp(cx - cw // 2, 0, W); x2 = torch.clamp(cx + cw // 2, 0, W)
    y1 = torch.clamp(cy - ch // 2, 0, H); y2 = torch.clamp(cy + ch // 2, 0, H)
    x[:, :, x1:x2, y1:y2] = x[idx, :, x1:x2, y1:y2]
    real_lam = 1 - ((x2 - x1) * (y2 - y1)).float() / (W * H)
    return x, y, y[idx], real_lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def accuracy_gpu(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        return [correct[:k].reshape(-1).float().sum(0).mul_(100.0 / target.size(0))
                for k in topk]

def attention_map(feat):
    """Spatial attention map (Zagoruyko & Komodakis 2017). (B,C,H,W) -> (B,H*W) normalized."""
    return F.normalize(feat.pow(2).mean(dim=1).flatten(1), p=2, dim=1)


def feature_kd_loss(s_feat, t_feat):
    """Attention Transfer loss (teacher detached inside)."""
    return (attention_map(s_feat) - attention_map(t_feat.detach())).pow(2).mean()


def get_mixup_decay(epoch, total_ep, mode, init_a, min_a):
    if mode in ("constant", "none"):
        return init_a
    p = min(epoch / max(total_ep, 1), 1.0)
    if mode == "cosine":
        return min_a + 0.5 * (init_a - min_a) * (1 + math.cos(math.pi * p))
    if mode == "linear":
        return init_a + (min_a - init_a) * p
    if mode == "step":
        step = max(total_ep // 4, 1)
        return max(init_a * (0.5 ** (epoch // step)), min_a)
    if mode == "random":
        return random.uniform(min_a, init_a)
    return init_a

def train_epoch(model, loader, criterion, optimizer, scaler, gpu_tf,
                alpha, epoch, distill_start_ep,
                teacher=None, kd_T=4.0, kd_w=0.0,
                grad_clip=None, clip_binary=None, ema=None,
                feature_dict=None, feature_kd_w=0.0):
    """`clip_binary`: list of A1W1BinaryConv2dV2 modules (cached in _run_phase)
    to clamp per step. None/empty → skip."""
    model.train()
    n, loss_sum, t1_sum, t5_sum = 0, 0.0, 0.0, 0.0
    beta = torch.distributions.Beta(alpha, alpha) if alpha > 0 else None

    for x, y in loader:
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        x = gpu_tf(x); B = y.size(0)
        x = x.contiguous(memory_format=torch.channels_last)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            use_mix = beta is not None
            if use_mix:
                lam = beta.sample().to(device)
                if random.random() < 0.5:
                    xm, y_a, y_b, lam = mixup_data(x, y, lam)
                else:
                    xm, y_a, y_b, lam = cutmix_data(x, y, lam)
                out = model(xm)
                base_loss = mixup_criterion(criterion, out, y_a, y_b, lam)
            else:
                out = model(x)
                base_loss = criterion(out, y)
                xm = x

            loss = base_loss
            if teacher is not None and kd_w > 0 and epoch >= distill_start_ep:
                with torch.inference_mode():
                    t_out = teacher(xm)
                distill = F.kl_div(
                    F.log_softmax(out / kd_T, dim=1),
                    F.softmax(t_out / kd_T, dim=1),
                    reduction='batchmean'
                ) * (kd_T * kd_T)
                loss = (1.0 - kd_w) * base_loss + kd_w * distill

                # Attention Transfer feature KD on layer4 with linear ramp-up.
                if feature_dict is not None and feature_kd_w > 0:
                    s_feat = feature_dict.get('student')
                    t_feat = feature_dict.get('teacher')
                    if s_feat is not None and t_feat is not None:
                        ramp = min(1.0, max(0, epoch - distill_start_ep + 1) / 10.0)
                        loss = loss + feature_kd_w * ramp * feature_kd_loss(s_feat, t_feat)

            with torch.no_grad():
                if use_mix:
                    _, pred = out.topk(5, 1, True, True); pred = pred.t()
                    ca = pred.eq(y_a.view(1, -1).expand_as(pred))
                    cb = pred.eq(y_b.view(1, -1).expand_as(pred))
                    a1 = (lam * ca[:1].sum() + (1 - lam) * cb[:1].sum()) / B * 100
                    a5 = (lam * ca[:5].sum() + (1 - lam) * cb[:5].sum()) / B * 100
                else:
                    accs = accuracy_gpu(out, y, topk=(1, 5))
                    a1, a5 = accs[0], accs[1]

        scaler.scale(loss).backward()
        if grad_clip is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer); scaler.update()

        if clip_binary:
            for m in clip_binary:
                m.weight.data.clamp_(-1.0, 1.0)

        if ema is not None:
            ema.update()

        loss_sum += loss.detach().item() * B
        t1_sum   += a1.detach().item() * B
        t5_sum   += a5.detach().item() * B
        n        += B

    return loss_sum / n, t1_sum / n, t5_sum / n

@torch.inference_mode()
def evaluate(model, loader, criterion, gpu_tf):
    model.eval()
    n, loss_sum, t1_sum, t5_sum = 0, 0.0, 0.0, 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        x = gpu_tf(x)
        x = x.contiguous(memory_format=torch.channels_last)
        out = model(x)
        loss = criterion(out, y)
        a1, a5 = accuracy_gpu(out, y, topk=(1, 5))
        B = y.size(0)
        loss_sum += loss.item() * B
        t1_sum   += a1.item() * B
        t5_sum   += a5.item() * B
        n        += B
    return loss_sum / n, t1_sum / n, t5_sum / n


In [ ]:
# ============================================================
# Teacher backbone + optimizer factory
# ============================================================
def _init_weights(model):
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

def build_teacher():
    """ResNet18 ImageNet pretrained → CIFAR-100 stem adaptation."""
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    # CIFAR stem: 3x3 stride 1, drop maxpool
    model.conv1.stride = (1, 1)
    model.maxpool = nn.Identity()
    # Re-init classifier for 100 classes
    in_f = model.fc.in_features
    model.fc = nn.Linear(in_f, 100)
    _init_weights(model.fc)
    return model.to(device)

def make_optimizer(model, lr, weight_decay):
    """AdamW with no-decay on biases & 1-D params (BN, PReLU, RPReLU γ/β, RSign θ)."""
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (no_decay if (p.dim() == 1 or n.endswith('.bias')) else decay).append(p)
    return optim.AdamW([
        {"params": decay,    "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ], lr=lr)


In [ ]:
# ============================================================
# Binary primitives (used by A1W1ResNet18v2)
# ============================================================
class BinaryActivationSTE(torch.autograd.Function):
    """sign(x) with clipped straight-through estimator (|x|<=1)."""
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.ge(0).to(x.dtype).mul(2).sub(1)
    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * (x.abs() <= 1).to(grad_output.dtype)

class BinaryWeightSTE(torch.autograd.Function):
    """sign(w) with pure identity STE."""
    @staticmethod
    def forward(ctx, w):
        return w.ge(0).to(w.dtype).mul(2).sub(1)
    @staticmethod
    def backward(ctx, g):
        return g

class RSignActivation(nn.Module):
    """Sign activation with per-channel learnable threshold (ReActNet RSign)."""
    def __init__(self, channels):
        super().__init__()
        self.threshold = nn.Parameter(torch.zeros(1, channels, 1, 1))
    def forward(self, x):
        return BinaryActivationSTE.apply(x - self.threshold.to(x.dtype))

class ShortcutDownsample(nn.Module):
    """Parameter-free shortcut: avgpool + channel zero-pad (BNN-friendly)."""
    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.pool = nn.AvgPool2d(stride) if stride > 1 else nn.Identity()
        self.pad_ch = out_ch - in_ch
    def forward(self, x):
        x = self.pool(x)
        if self.pad_ch <= 0:
            return x
        return F.pad(x, (0, 0, 0, 0, 0, self.pad_ch))


In [ ]:
# ============================================================
# A1W1 ReActNet-lite v2 — RPReLU + double-skip + detached α + AMP-safe
# ============================================================
class RPReLU(nn.Module):
    """ReActNet RPReLU: PReLU(x - γ) + β, per-channel learnable shift+slope+bias."""
    def __init__(self, channels):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.beta  = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.prelu = nn.PReLU(num_parameters=channels)
    def forward(self, x):
        return self.prelu(x - self.gamma) + self.beta

class A1W1BinaryConv2dV2(nn.Module):
    """1-bit conv: binary weight × binary activation, per-channel detached α scale.
    Runs in autocast dtype (fp16 during training) for Tensor-Core speedup.
    ReActNet original training also uses mixed precision — sign/STE boundaries
    are stable enough in fp16 in practice."""
    def __init__(self, in_ch, out_ch, k, stride=1, padding=0, bias=False):
        super().__init__()
        self.stride, self.padding = stride, padding
        self.weight = nn.Parameter(torch.empty(out_ch, in_ch, k, k))
        self.bias   = nn.Parameter(torch.zeros(out_ch)) if bias else None
        nn.init.kaiming_normal_(self.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x):
        w_b   = BinaryWeightSTE.apply(self.weight)
        alpha = self.weight.detach().abs().mean(dim=[1, 2, 3]).view(1, -1, 1, 1)
        out   = F.conv2d(x, w_b, self.bias, self.stride, self.padding)
        return out * alpha.to(out.dtype)

class A1W1BasicBlockV2(nn.Module):
    """Block: BN→RSign→BinConv→BN→RPReLU → +residual_a → BN→RSign→BinConv→BN→RPReLU → +residual_b.
    `use_double_skip=False` removes the second identity skip (single-skip variant)."""
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, use_rprelu=True, use_double_skip=True):
        super().__init__()
        self.use_double_skip = use_double_skip
        self.bn1    = nn.BatchNorm2d(in_planes)
        self.rsign1 = RSignActivation(in_planes)
        self.conv1  = A1W1BinaryConv2dV2(in_planes, planes, 3, stride=stride, padding=1)
        self.bn2    = nn.BatchNorm2d(planes)
        self.act1   = RPReLU(planes) if use_rprelu else nn.Identity()

        self.bn3    = nn.BatchNorm2d(planes)
        self.rsign2 = RSignActivation(planes)
        self.conv2  = A1W1BinaryConv2dV2(planes, planes, 3, padding=1)
        self.bn4    = nn.BatchNorm2d(planes)
        self.act2   = RPReLU(planes) if use_rprelu else nn.Identity()

        self.downsample_a = None
        if stride != 1 or in_planes != planes:
            self.downsample_a = ShortcutDownsample(in_planes, planes, stride)

    def forward(self, x):
        # Branch 1
        res_a = x if self.downsample_a is None else self.downsample_a(x)
        out = self.act1(self.bn2(self.conv1(self.rsign1(self.bn1(x)))))
        out = out + res_a
        # Branch 2 (optional double-skip)
        res_b = out if self.use_double_skip else 0
        out2 = self.act2(self.bn4(self.conv2(self.rsign2(self.bn3(out)))))
        return out2 + res_b

class A1W1ResNet18v2(nn.Module):
    """ResNet18-style ReActNet-lite v2. Stem conv & classifier stay full-precision."""
    def __init__(self, num_classes=100, use_rprelu=True, use_double_skip=True):
        super().__init__()
        self.use_rprelu = use_rprelu
        self.use_double_skip = use_double_skip
        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)
        self.act_in = RPReLU(64) if use_rprelu else nn.PReLU(64)
        self.layer1 = self._make_layer(64,  64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128,256, 2, stride=2)
        self.layer4 = self._make_layer(256,512, 2, stride=2)
        self.bn_out = nn.BatchNorm2d(512)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_classes)
        self._init_weights()

    def _make_layer(self, in_p, p, n, stride):
        layers = [A1W1BasicBlockV2(in_p, p, stride,
                                   use_rprelu=self.use_rprelu,
                                   use_double_skip=self.use_double_skip)]
        for _ in range(1, n):
            layers.append(A1W1BasicBlockV2(p, p,
                                           use_rprelu=self.use_rprelu,
                                           use_double_skip=self.use_double_skip))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (A1W1BinaryConv2dV2, nn.Conv2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if getattr(m, 'bias', None) is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.act_in(self.bn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.bn_out(x); x = self.avgpool(x)
        return self.fc(torch.flatten(x, 1))

def init_student_from_teacher(student, teacher):
    """Copy any state-dict tensor that matches by name + shape (stem conv/bn,
    block conv1/bn1/bn2/conv2). Threshold/RPReLU/bn3/bn4 stay at student init."""
    sb = student.module if isinstance(student, nn.DataParallel) else student
    tb = teacher.module if isinstance(teacher, nn.DataParallel) else teacher
    s_state = sb.state_dict()
    t_state = tb.state_dict()
    copied = 0
    new_state = OrderedDict()
    for k, v in s_state.items():
        ref = t_state.get(k)
        if ref is not None and ref.shape == v.shape:
            new_state[k] = ref.detach().clone()
            copied += 1
        else:
            new_state[k] = v
    sb.load_state_dict(new_state, strict=True)
    return copied, len(s_state)


In [ ]:
# ============================================================
# EMA (exponential moving average) of student + KD teacher wrapper
# ============================================================
class EMA:
    """Maintains an EMA shadow of `student` in `shadow` (decay closer to 1 = slower)."""
    def __init__(self, student, shadow, decay=0.999):
        self.student = student
        self.shadow  = shadow
        self.decay   = decay
        # Initialise shadow == student (strip DataParallel prefix if needed).
        s_state = self.student.state_dict()
        clean   = {k.replace('module.', ''): v for k, v in s_state.items()}
        self.shadow.load_state_dict(clean)

    @torch.no_grad()
    def update(self):
        s_state = self.student.state_dict()
        for k, v in self.shadow.state_dict().items():
            s_key = 'module.' + k if 'module.' + k in s_state else k
            v.copy_(self.decay * v + (1.0 - self.decay) * s_state[s_key])

class DistillationWrapper:
    """Frozen teacher exposed via .teacher for the train loop."""
    def __init__(self, teacher):
        self.teacher = teacher
        self.teacher.eval()
        for p in self.teacher.parameters():
            p.requires_grad_(False)


In [ ]:
# ============================================================
# Pipeline: Phase A (teacher quick FT) → Phase B (A1W1 student + KD)
# ============================================================
def _run_phase(model, trainloader, valloader, criterion, scaler,
               train_tf, val_tf, total_epochs, base_lr, weight_decay,
               warmup_eps, init_alpha, min_alpha,
               teacher=None, kd_T=4.0, kd_w=0.0, distill_start_ep=0,
               grad_clip=None, clip_binary=False, best_ckpt="best.pth",
               phase_name="phase", use_ema=True,
               feature_kd_w=0.0):
    optimizer = make_optimizer(model, base_lr, weight_decay)
    warmup_sch = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=max(warmup_eps, 1)
    )
    cosine_sch = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(total_epochs - warmup_eps, 1), eta_min=min_lr
    )
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_sch, cosine_sch], milestones=[warmup_eps]
    )

    base = model.module if isinstance(model, nn.DataParallel) else model
    # Cache binary conv modules once so train_epoch can clamp without re-walking
    # the module tree every step.
    binary_convs = (
        [m for m in base.modules() if m.__class__.__name__ == "A1W1BinaryConv2dV2"]
        if clip_binary else None
    )
    # EMA shadow only when use_ema=True. Disable for binary nets — sign() of
    # weight-averaged binary weights is effectively random, so EMA collapses
    # to ~chance accuracy. (BNN papers ReActNet/Bi-Real/ReCU/IR-Net don't use EMA.)
    if use_ema:
        shadow = copy.deepcopy(base).to(device).eval()
        for p in shadow.parameters():
            p.requires_grad_(False)
        ema = EMA(model, shadow, decay=ema_decay)
    else:
        shadow = None
        ema = None

    # Feature KD (AT) hooks on layer4. Active only when feature_kd_w>0 and teacher present.
    features = {'student': None, 'teacher': None}
    h_s = h_t = None
    if feature_kd_w > 0 and teacher is not None:
        student_layer4 = (model.module if isinstance(model, nn.DataParallel) else model).layer4
        teacher_layer4 = (teacher.module if isinstance(teacher, nn.DataParallel) else teacher).layer4
        h_s = student_layer4.register_forward_hook(lambda m, i, o: features.__setitem__('student', o))
        h_t = teacher_layer4.register_forward_hook(lambda m, i, o: features.__setitem__('teacher', o))

    best_val = 0.0
    history = {
        "epoch": [], "train_loss": [], "train_top1": [], "train_top5": [],
        "val_loss":   [], "val_top1":   [], "val_top5":   [],
        "val_top1_ema": [],
        "lr": [], "alpha": [],
    }
    for epoch in range(total_epochs):
        t0 = time.time()
        alpha = get_mixup_decay(epoch, total_epochs, mixup_decay, init_alpha, min_alpha)

        train_loss, t1, t5 = train_epoch(
            model, trainloader, criterion, optimizer, scaler, train_tf,
            alpha=alpha, epoch=epoch, distill_start_ep=distill_start_ep,
            teacher=teacher, kd_T=kd_T, kd_w=kd_w,
            grad_clip=grad_clip, clip_binary=binary_convs,
            ema=ema,
            feature_dict=features if (h_s is not None) else None,
            feature_kd_w=feature_kd_w,
        )

        val_loss, v1, v5 = evaluate(model, valloader, criterion, val_tf)
        if use_ema and epoch >= warmup_eps:
            _, v1_ema, _ = evaluate(shadow, valloader, criterion, val_tf)
        else:
            v1_ema = None
        scheduler.step()

        # Pick the reference val for best-ckpt: EMA when meaningful, otherwise live.
        ref = v1_ema if (use_ema and epoch >= warmup_eps) else v1
        if ref > best_val:
            best_val = ref
            target = shadow if (use_ema and epoch >= warmup_eps) else base
            torch.save(target.state_dict(), best_ckpt)

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(float(train_loss))
        history["train_top1"].append(float(t1))
        history["train_top5"].append(float(t5))
        history["val_loss"].append(float(val_loss))
        history["val_top1"].append(float(v1))
        history["val_top5"].append(float(v5))
        history["val_top1_ema"].append(float(v1_ema) if v1_ema is not None else float("nan"))
        history["lr"].append(float(optimizer.param_groups[0]["lr"]))
        history["alpha"].append(float(alpha))

        dur = time.time() - t0
        ema_str = f"ema {v1_ema:.2f}%" if v1_ema is not None else "ema --"
        print(f"[{phase_name}] {epoch+1:>3}/{total_epochs} | "
              f"tr {train_loss:.3f}/{t1:.2f}% | val {v1:.2f}% ({ema_str}) | "
              f"best {best_val:.2f}% | lr {optimizer.param_groups[0]['lr']:.6f} | "
              f"α {alpha:.2f} | {dur:.1f}s")

    # ---- Save per-epoch history (CSV + pickle) ----
    hist_csv = f"history_{phase_name.lower()}_{exp_id}.csv"
    hist_pkl = f"history_{phase_name.lower()}_{exp_id}.pkl"
    pd.DataFrame(history).to_csv(hist_csv, index=False)
    with open(hist_pkl, "wb") as f:
        pickle.dump(history, f)
    print(f"[{phase_name}] history saved: {hist_csv} / {hist_pkl}")

    if h_s is not None: h_s.remove()
    if h_t is not None: h_t.remove()

    return best_val, (shadow if use_ema else base)

def main_pipeline():
    set_seed(seed)
    scaler = torch.amp.GradScaler('cuda')

    trainloader, valloader = train_val_loader()
    train_tf, val_tf = get_gpu_transforms()

    # ---- Phase A: quick teacher --------------------------------
    if skip_phase_a_if_ckpt and os.path.exists(teacher_ckpt_path):
        print("\n" + "=" * 60)
        print(f">>> Phase A skipped: loading teacher from '{teacher_ckpt_path}'")
        print("=" * 60)
        teacher_shadow = build_teacher()
        sd = torch.load(teacher_ckpt_path, map_location=device)
        sd = {k.replace('module.', ''): v for k, v in sd.items()}
        teacher_shadow.load_state_dict(sd)
        teacher_shadow = teacher_shadow.to(memory_format=torch.channels_last)
        teacher_shadow.eval()
        # Quick eval to report stored teacher quality.
        criterion_T = nn.CrossEntropyLoss(label_smoothing=teacher_label_smoothing)
        _, t_top1, t_top5 = evaluate(teacher_shadow, valloader, criterion_T, val_tf)
        print(f">>> Loaded teacher val Top-1={t_top1:.2f}%, Top-5={t_top5:.2f}%")
        # Save a 1-row history for the loaded teacher so plotting can reference it
        loaded_hist = {
            "epoch":[0], "train_loss":[float('nan')], "train_top1":[float('nan')], "train_top5":[float('nan')],
            "val_loss":[float('nan')], "val_top1":[float(t_top1)], "val_top5":[float(t_top5)],
            "val_top1_ema":[float(t_top1)], "lr":[float('nan')], "alpha":[float('nan')],
        }
        pd.DataFrame(loaded_hist).to_csv(f"history_tch_{exp_id}.csv", index=False)
        with open(f"history_tch_{exp_id}.pkl", "wb") as f:
            pickle.dump(loaded_hist, f)
    else:
        print("\n" + "=" * 60)
        print(">>> Phase A: ResNet18 (ImageNet pretrained) → CIFAR-100 FT")
        print("=" * 60)
        teacher = build_teacher().to(memory_format=torch.channels_last)
        if torch.cuda.device_count() > 1:
            teacher = nn.DataParallel(teacher)
        criterion_T = nn.CrossEntropyLoss(label_smoothing=teacher_label_smoothing)
        teacher_best, teacher_shadow = _run_phase(
            teacher, trainloader, valloader, criterion_T, scaler, train_tf, val_tf,
            total_epochs=teacher_epochs, base_lr=teacher_lr,
            weight_decay=teacher_weight_decay,
            warmup_eps=teacher_warmup_epochs,
            init_alpha=teacher_mixup_alpha, min_alpha=teacher_min_alpha,
            teacher=None, kd_w=0.0, grad_clip=None, clip_binary=False,
            best_ckpt=teacher_ckpt_path, phase_name="Tch",
        )
        print(f">>> Phase A done — teacher EMA best = {teacher_best:.2f}%")

    # ---- Phase B: A1W1 student + KD ----------------------------
    print("\n" + "=" * 60)
    print(">>> Phase B: A1W1ResNet18v2 student + KD")
    print("=" * 60)
    student = A1W1ResNet18v2(num_classes=100,
                             use_rprelu=student_use_rprelu,
                             use_double_skip=student_use_double_skip).to(device)
    copied, total = init_student_from_teacher(student, teacher_shadow)
    print(f"    [init] copied {copied}/{total} tensors from teacher EMA")
    student = student.to(memory_format=torch.channels_last)
    # Skip DataParallel for student: simpler Feature-KD hooks, and a single T4
    # easily fits ResNet18 fp16-binary at batch 1024.
    criterion_S = nn.CrossEntropyLoss(label_smoothing=student_label_smoothing)
    teacher_for_kd = DistillationWrapper(teacher_shadow).teacher

    student_best, _ = _run_phase(
        student, trainloader, valloader, criterion_S, scaler, train_tf, val_tf,
        total_epochs=student_epochs, base_lr=student_lr,
        weight_decay=student_weight_decay,
        warmup_eps=student_warmup_epochs,
        init_alpha=student_mixup_alpha, min_alpha=student_min_alpha,
        teacher=teacher_for_kd, kd_T=student_kd_T, kd_w=student_kd_weight,
        distill_start_ep=student_warmup_epochs,
        grad_clip=student_grad_clip, clip_binary=True,
        best_ckpt=submission_ckpt_path, phase_name="Stu",
        use_ema=False,                            # EMA-on-binary collapses sign().
        feature_kd_w=student_feature_kd_w,        # AT distillation on layer4.
    )
    print(f"\n>>> Phase B done — student EMA best = {student_best:.2f}%")
    print(f">>> Submission checkpoint saved to '{submission_ckpt_path}'")
    return student_best

main_pipeline()


In [ ]:
# ============================================================
# Submission: load best student checkpoint and write submission.csv
# ============================================================
_, val_tf = get_gpu_transforms()

model = A1W1ResNet18v2(num_classes=100,
                       use_rprelu=student_use_rprelu,
                       use_double_skip=student_use_double_skip).to(device)
state = torch.load(submission_ckpt_path, map_location=device)
state = {k.replace('module.', ''): v for k, v in state.items()}
model.load_state_dict(state)
model = model.to(memory_format=torch.channels_last)
model.eval()

testloader = test_loader()
all_preds = []
with torch.inference_mode():
    for x in testloader:
        x = x.to(device, non_blocking=True)
        x = val_tf(x)
        x = x.contiguous(memory_format=torch.channels_last)
        all_preds.extend(model(x).argmax(dim=1).cpu().numpy())

submission = pd.read_csv(os.path.join(BASE_PATH, 'sample_submission.csv'))
submission['label'] = all_preds
submission.to_csv('submission.csv', index=False)
print("submission.csv written:", len(submission), "rows")
